In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import io
import json
import re
import subprocess
import sys
import time
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

if importlib.util.find_spec('astropy') is None or importlib.util.find_spec('astroquery') is None:
    print('[INFO] Installing Astropy and Astroquery...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'astropy', 'astroquery'], check=True)

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from astroquery.mast import Observations

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/day18-target-pixel-audit-gaia-retry')
TARGET = '11521787'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Find the source's coordinates in the attached 20K feature table.
data_candidates = []
for feature_path in INPUT.rglob('features.csv'):
    try:
        table = pd.read_csv(feature_path, dtype={'source_id': 'string'})
        if not {'source_id', 'ra_deg', 'dec_deg'}.issubset(table.columns):
            continue
        ids = table['source_id'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
        rows = table.loc[ids.eq(TARGET)].copy().dropna(subset=['ra_deg', 'dec_deg'])
        if not rows.empty:
            data_candidates.append((len(table), feature_path, rows))
    except Exception as exc:
        print(f'[INFO] Skipped unreadable feature file {feature_path}: {type(exc).__name__}')
if not data_candidates:
    raise FileNotFoundError('TIC 11521787 with RA/Dec was not found. Attach the Cosmic 20000 swan v2 notebook output.')
_, FEATURE_PATH, target_rows = max(data_candidates, key=lambda item: (item[0], str(item[1])))
ra_deg = float(target_rows['ra_deg'].dropna().iloc[0])
dec_deg = float(target_rows['dec_deg'].dropna().iloc[0])
sectors = sorted({int(value) for value in target_rows['sector'].dropna().tolist()}) if 'sector' in target_rows else []
print(f'[PASS] Feature input: {FEATURE_PATH}; observations={len(target_rows)}; sectors={sectors}')
print(f'[INFO] TIC {TARGET} ICRS position: RA={ra_deg:.8f}, Dec={dec_deg:.8f}')

# Query Gaia DR3 from ESA TAP first, then parse TSV or VOTable from several VizieR mirrors.
def _masked(value):
    try:
        return bool(np.ma.is_masked(value) or value is None)
    except Exception:
        return value is None

def _records_from_astropy_table(table, limit):
    # VizieR normally returns the requested Gaia DR3 column names. Match case-insensitively
    # and accept common aliases in case a mirror decorates or renames a column.
    by_lower = {str(name).strip().lower(): name for name in table.colnames}
    aliases = {
        'Source': ('source', 'source_id', 'gaia', 'gaia_dr3'),
        'RA_ICRS': ('ra_icrs', 'ra', 'raj2000', 'ra_deg'),
        'DE_ICRS': ('de_icrs', 'dec', 'dej2000', 'dec_deg'),
        'Gmag': ('gmag', 'phot_g_mean_mag', 'g_mean_mag'),
    }
    chosen = {}
    for standard, names in aliases.items():
        for name in names:
            if name in by_lower:
                chosen[standard] = by_lower[name]
                break
    if not {'Source', 'RA_ICRS', 'DE_ICRS'}.issubset(chosen):
        raise RuntimeError(f'VOTable columns do not include Gaia source/RA/Dec fields: {table.colnames}')
    records = []
    for row in table[:limit]:
        source, ra, dec = (row[chosen[k]] for k in ('Source', 'RA_ICRS', 'DE_ICRS'))
        if _masked(source) or _masked(ra) or _masked(dec):
            continue
        try:
            record = {'Source': int(source), 'RA_ICRS': float(ra), 'DE_ICRS': float(dec)}
        except (TypeError, ValueError, OverflowError):
            continue
        gmag = row[chosen['Gmag']] if 'Gmag' in chosen else None
        try:
            record['Gmag'] = None if _masked(gmag) else float(gmag)
        except (TypeError, ValueError, OverflowError):
            record['Gmag'] = None
        records.append(record)
    return records

def _records_from_response(response, limit):
    content_type = response.headers.get('Content-Type', '').lower()
    body_bytes = response.content
    body_text = response.text
    lower_start = body_text[:800].lower()
    if '<html' in lower_start or 'archive down for maintenance' in lower_start:
        raise RuntimeError(f'VizieR returned an HTML/error page: {body_text[:240]!r}')

    # VizieR mirrors may answer an ASU request with VOTable XML. Some responses prepend
    # MIME metadata before the XML; trim to the XML root before giving it to Astropy.
    xml_at = body_text.lower().find('<votable')
    if 'votable' in content_type or xml_at >= 0 or '<?xml' in lower_start:
        if xml_at >= 0:
            body_bytes = body_text[xml_at:].encode('utf-8')
        table = Table.read(io.BytesIO(body_bytes), format='votable')
        return _records_from_astropy_table(table, limit), 'VOTable'

    lines = [line for line in body_text.splitlines() if line.strip() and not line.lstrip().startswith('#')]
    parsed = [next(csv.reader([line], delimiter='\t')) for line in lines]
    header_i = next((i for i, row in enumerate(parsed)
                     if {'Source', 'RA_ICRS', 'DE_ICRS'}.issubset({cell.strip() for cell in row})), None)
    if header_i is None:
        raise RuntimeError(f'No Gaia TSV header or VOTable XML found; content-type={content_type!r}; preview={body_text[:260]!r}')
    header = [cell.strip() for cell in parsed[header_i]]
    raw_table = []
    for row in parsed[header_i + 1:]:
        if len(row) == len(header):
            raw_table.append(dict(zip(header, [cell.strip() for cell in row])))
    records = []
    for item in raw_table[:limit]:
        try:
            record = {'Source': int(item['Source']), 'RA_ICRS': float(item['RA_ICRS']), 'DE_ICRS': float(item['DE_ICRS'])}
        except (KeyError, TypeError, ValueError, OverflowError):
            continue
        try:
            record['Gmag'] = float(item['Gmag'])
        except (KeyError, TypeError, ValueError, OverflowError):
            record['Gmag'] = None
        records.append(record)
    return records, 'TSV'

def query_gaia(ra, dec, radius_arcsec=60.0, max_rows=1000):
    errors = []
    center = SkyCoord(ra*u.deg, dec*u.deg, frame='icrs')
    try:
        from astroquery.gaia import Gaia
        job = Gaia.cone_search_async(
            center, radius=radius_arcsec*u.arcsec,
            columns=['source_id', 'ra', 'dec', 'phot_g_mean_mag'], verbose=False,
        )
        table = job.get_results()
        records = []
        for row in table[:max_rows]:
            sid, raval, decval, gval = (row[k] for k in ('source_id', 'ra', 'dec', 'phot_g_mean_mag'))
            if _masked(sid) or _masked(raval) or _masked(decval):
                continue
            records.append({'Source': int(sid), 'RA_ICRS': float(raval), 'DE_ICRS': float(decval),
                            'Gmag': None if _masked(gval) else float(gval)})
        return {
            'status': 'query_succeeded', 'catalog': 'Gaia DR3 via ESA Gaia TAP',
            'queried_at_utc': datetime.now(timezone.utc).isoformat(),
            'center_icrs_deg': {'ra': ra, 'dec': dec}, 'radius_arcsec': radius_arcsec,
            'record_count_returned': len(records), 'records': records,
            'source_url': 'https://gea.esac.esa.int/archive/', 'attempt_errors': errors,
        }
    except Exception as exc:
        errors.append({'service': 'ESA Gaia TAP', 'error': f'{type(exc).__name__}: {str(exc)[:400]}'})
        print(f'[WARN] Gaia TAP query failed: {errors[-1]["error"]}')

    params = {
        '-source': 'I/355/gaiadr3', '-c.ra': f'{ra:.8f}', '-c.dec': f'{dec:.8f}',
        '-c.rs': str(radius_arcsec), '-out': 'Source,RA_ICRS,DE_ICRS,Gmag',
        '-out.max': str(max_rows), '-out.meta': 'hu',
    }
    mirrors = [
        'https://vizier.cfa.harvard.edu/viz-bin/asu-tsv',
        'https://vizier.cds.unistra.fr/viz-bin/asu-tsv',
        'https://vizier.nao.ac.jp/viz-bin/asu-tsv',
    ]
    for mirror in mirrors:
        try:
            response = requests.get(
                mirror, params=params,
                headers={'User-Agent': 'CosmicSwanPixelAudit/3.0', 'Accept': 'text/tab-separated-values, application/x-votable+xml, */*'},
                timeout=(8, 22),
            )
            response.raise_for_status()
            records, response_format = _records_from_response(response, max_rows)
            return {
                'status': 'query_succeeded', 'catalog': f'Gaia DR3 via VizieR {response_format} fallback',
                'queried_at_utc': datetime.now(timezone.utc).isoformat(),
                'center_icrs_deg': {'ra': ra, 'dec': dec}, 'radius_arcsec': radius_arcsec,
                'record_count_returned': len(records), 'records': records,
                'http_status': response.status_code, 'source_url': response.url,
                'response_format': response_format, 'attempt_errors': errors,
            }
        except Exception as exc:
            errors.append({'service': mirror, 'error': f'{type(exc).__name__}: {str(exc)[:450]}'})
            print(f'[WARN] VizieR fallback failed ({mirror}): {errors[-1]["error"]}')

    return {
        'status': 'query_failed', 'catalog': 'Gaia DR3',
        'queried_at_utc': datetime.now(timezone.utc).isoformat(),
        'center_icrs_deg': {'ra': ra, 'dec': dec}, 'radius_arcsec': radius_arcsec,
        'record_count_returned': None, 'records': [], 'attempt_errors': errors,
        'interpretation': 'Gaia and all configured VizieR mirror requests failed; this is not a zero-neighbor result.',
    }

# Download TESS target-pixel products only for the sectors represented in the saved target features.
mast_status={'status':'query_started','sectors_requested':sectors}
downloaded=[]
try:
    center=SkyCoord(ra_deg*u.deg,dec_deg*u.deg,frame='icrs')
    observations=Observations.query_region(center,radius=3.0*u.arcsec)
    if observations is None or len(observations)==0:
        raise RuntimeError('MAST returned no observations near this TIC')
    if 'obs_collection' not in observations.colnames:
        raise RuntimeError('MAST response is missing obs_collection; cannot safely select TESS products')
    collections=np.char.upper(np.asarray(observations['obs_collection']).astype(str))
    observations=observations[collections=='TESS']
    if len(observations)==0:
        raise RuntimeError('No TESS observations were returned near this TIC')
    products=Observations.get_product_list(observations)
    frame=products.to_pandas()
    filenames=frame['productFilename'].astype('string')
    is_tpf=filenames.str.lower().str.endswith('tp.fits', na=False)
    if 'productSubGroupDescription' in frame:
        subgroup=frame['productSubGroupDescription'].astype('string').str.upper()
        is_tpf &= subgroup.eq('TP') | subgroup.isna()
    if sectors:
        sector_matches=filenames.apply(lambda name: any(re.search(fr'-s{sector:04d}-',str(name),re.I) for sector in sectors))
        is_tpf &= sector_matches
    selected=products[np.flatnonzero(is_tpf.to_numpy())]
    if len(selected)==0:
        raise RuntimeError(f'MAST returned no target-pixel products for sectors {sectors}')
    raw_dir=OUTPUT/'raw'/'target-pixels'
    raw_dir.mkdir(parents=True,exist_ok=True)
    print(f'[INFO] Downloading {len(selected)} target-pixel products from MAST...',flush=True)
    download_table=Observations.download_products(selected,download_dir=str(raw_dir),flat=True,cache=True,verbose=False)
    for row in download_table:
        local=Path(str(row['Local Path']))
        if not local.is_absolute():
            local=Path.cwd()/local
        status=str(row['Status']).strip().upper()
        if status in {'COMPLETE','SKIPPED'} and local.is_file():
            downloaded.append(local)
    if not downloaded:
        raise RuntimeError('MAST did not return any completed local target-pixel files')
    mast_status={'status':'query_succeeded','observations_near_target':len(observations),'products_selected':len(selected),'products_downloaded':len(downloaded),'sectors_requested':sectors}
    print(f'[PASS] TESS target-pixel products downloaded: {len(downloaded)}')
except Exception as exc:
    mast_status={'status':'query_failed','sectors_requested':sectors,'error':f'{type(exc).__name__}: {str(exc)[:400]}'}
    print(f"[FAIL] MAST target-pixel download: {mast_status['error']}")

print('[INFO] Querying Gaia DR3 neighbors around the target...',flush=True)
gaia=query_gaia(ra_deg,dec_deg)
print(f"[{'PASS' if gaia['status']=='query_succeeded' else 'WARN'}] Gaia: {gaia['status']}; rows={gaia.get('record_count_returned')}")

def audit_product(path, gaia_result):
    with fits.open(path,memmap=True) as hdul:
        if len(hdul)<3 or hdul[1].data is None or hdul[2].data is None:
            raise ValueError('TPF is missing expected flux/aperture extensions')
        data=hdul[1].data
        names={name.upper():name for name in data.names}
        if 'FLUX' not in names:
            raise ValueError('TPF has no FLUX field')
        flux=np.asarray(data[names['FLUX']],dtype=np.float64)
        if flux.ndim!=3:
            raise ValueError(f'Unexpected FLUX cube shape: {flux.shape}')
        if 'FLUX_BKG' in names:
            flux=flux-np.asarray(data[names['FLUX_BKG']],dtype=np.float64)
        aperture=np.asarray(hdul[2].data)
        mask=(aperture.astype(np.int64)&2)>0
        if not mask.any():
            raise ValueError('No optimal photometric aperture pixels found (TESS APERTURE bit value 2)')
        image=np.nanmedian(flux,axis=0)
        positive=np.clip(image,0,None)
        inside_flux=float(np.nansum(positive[mask]))
        outside_flux=float(np.nansum(positive[~mask]))
        total=inside_flux+outside_flux
        header=hdul[2].header
        wcs=WCS(header)
        ra_obj=header.get('RA_OBJ',hdul[0].header.get('RA_OBJ',ra_deg))
        dec_obj=header.get('DEC_OBJ',hdul[0].header.get('DEC_OBJ',dec_deg))
        tx,ty=wcs.world_to_pixel_values(float(ra_obj),float(dec_obj))
        quality=np.asarray(data[names['QUALITY']],dtype=np.int64) if 'QUALITY' in names else np.zeros(flux.shape[0],dtype=np.int64)
        sector=int(header.get('SECTOR',hdul[0].header.get('SECTOR',0)) or 0)
        camera=int(header.get('CAMERA',hdul[0].header.get('CAMERA',0)) or 0)
        ccd=int(header.get('CCD',hdul[0].header.get('CCD',0)) or 0)
        shape=mask.shape
    neighbors=[]
    if gaia_result.get('status')=='query_succeeded':
        for record in gaia_result.get('records',[]):
            try:
                ra=float(record['RA_ICRS']); dec=float(record['DE_ICRS'])
                x,y=wcs.world_to_pixel_values(ra,dec)
            except Exception:
                continue
            xn,yn=int(round(x)),int(round(y))
            gmag=None
            try: gmag=float(record.get('Gmag'))
            except (TypeError,ValueError): pass
            separation=float(np.hypot(x-tx,y-ty))
            neighbors.append({
                'source_id':record.get('Source'),'ra_deg':ra,'dec_deg':dec,'g_magnitude':gmag,
                'x_pixel':float(x),'y_pixel':float(y),'separation_from_target_pixels':separation,
                'inside_tpf_cutout':bool(0<=yn<shape[0] and 0<=xn<shape[1]),
                'nearest_pixel_in_optimal_aperture':bool(0<=yn<shape[0] and 0<=xn<shape[1] and mask[yn,xn]),
                'likely_target_match':bool(separation<0.5),
            })
    return {
        'file':str(path.resolve()),'sector':sector,'camera':camera,'ccd':ccd,
        'cadences':int(flux.shape[0]),'pixels_per_frame':int(shape[0]*shape[1]),
        'optimal_aperture_pixels':int(mask.sum()),
        'aperture_positive_flux_fraction_of_cutout':inside_flux/total if total>0 else None,
        'outside_aperture_positive_flux_fraction_of_cutout':outside_flux/total if total>0 else None,
        'quality_zero_cadence_fraction':float(np.mean(quality==0)),
        'target_pixel':{'x_zero_based':float(tx),'y_zero_based':float(ty)},
        'aperture_mask':mask.astype(int).tolist(),
        'median_flux_image':image.tolist(),
        'gaia_neighbors_within_query_radius':neighbors,
    }

product_results=[]
for i,path in enumerate(downloaded,1):
    try:
        product_results.append(audit_product(path,gaia))
        print(f"[PASS] Audited product {i}/{len(downloaded)}: {path.name}")
    except Exception as exc:
        product_results.append({'file':str(path),'status':'audit_failed','error':f'{type(exc).__name__}: {str(exc)[:300]}'})

def clean_json(value):
    if isinstance(value,dict): return {str(key):clean_json(item) for key,item in value.items()}
    if isinstance(value,(list,tuple)): return [clean_json(item) for item in value]
    if isinstance(value,np.generic): value=value.item()
    if isinstance(value,float) and not np.isfinite(value): return None
    return value

report={
    'created_utc':datetime.now(timezone.utc).isoformat(),'source_id':TARGET,
    'coordinates_icrs_deg':{'ra':ra_deg,'dec':dec_deg},'feature_sectors':sectors,
    'mast_query':mast_status,'gaia_query':gaia,'target_pixel_products':product_results,
    'review_status':'screened_by_proxy; manually inspect the plotted TPF cutouts and Gaia associations',
    'interpretation_limits':[
        'The optimal aperture uses TESS APERTURE bit value 2; flux fractions are screening proxies, not calibrated contamination probabilities.',
        'Gaia completeness, WCS, PRF wings, and the TIC-to-Gaia association require human review.',
        'A failed MAST or VizieR request is not evidence that there are no contaminants.',
        'This one-target audit is not a complete pixel-level detector blind-spot survey.',
    ],
}
report_path=OUTPUT/'day18_target_pixel_audit.json'
report_path.write_text(json.dumps(clean_json(report),indent=2,allow_nan=False),encoding='utf-8')

summary_rows=[]
neighbor_rows=[]
for item in product_results:
    if item.get('status')=='audit_failed':
        summary_rows.append({'source_id':TARGET,'file':Path(item['file']).name,'status':'audit_failed','error':item.get('error')})
        continue
    summary_rows.append({
        'source_id':TARGET,'sector':item.get('sector'),'camera':item.get('camera'),'ccd':item.get('ccd'),
        'file':Path(item.get('file','')).name,'status':'screened','cadences':item.get('cadences'),
        'aperture_pixels':item.get('optimal_aperture_pixels'),
        'aperture_flux_fraction':item.get('aperture_positive_flux_fraction_of_cutout'),
        'outside_aperture_flux_fraction':item.get('outside_aperture_positive_flux_fraction_of_cutout'),
        'quality_zero_cadence_fraction':item.get('quality_zero_cadence_fraction'),
        'gaia_neighbors_in_aperture':sum(n.get('nearest_pixel_in_optimal_aperture',False) for n in item.get('gaia_neighbors_within_query_radius',[])),
    })
    for neighbor in item.get('gaia_neighbors_within_query_radius',[]):
        neighbor_rows.append({'sector':item.get('sector'),**neighbor})
summary_csv=OUTPUT/'day18_target_pixel_summary.csv'
neighbors_csv=OUTPUT/'day18_gaia_neighbors.csv'
pd.DataFrame(summary_rows).to_csv(summary_csv,index=False)
pd.DataFrame(neighbor_rows).to_csv(neighbors_csv,index=False)

plot_items=[item for item in product_results if item.get('status')!='audit_failed'][:6]
plot_path=OUTPUT/'day18_target_pixel_cutouts.png'
if plot_items:
    fig,axes=plt.subplots(1,len(plot_items),figsize=(5*len(plot_items),5),squeeze=False)
    for ax,item in zip(axes[0],plot_items):
        image=np.asarray(item['median_flux_image'],dtype=float)
        mask=np.asarray(item['aperture_mask'],dtype=float)
        finite=image[np.isfinite(image)]
        low,high=np.nanpercentile(finite,[5,99]) if finite.size else (0,1)
        ax.imshow(image,origin='lower',cmap='inferno',vmin=low,vmax=high)
        if mask.any(): ax.contour(mask,levels=[0.5],colors=['cyan'],linewidths=1)
        tp=item.get('target_pixel',{})
        ax.scatter(tp.get('x_zero_based'),tp.get('y_zero_based'),marker='+',s=120,c='white',linewidths=2)
        for neighbor in item.get('gaia_neighbors_within_query_radius',[]):
            if neighbor.get('inside_tpf_cutout'):
                ax.scatter(neighbor['x_pixel'],neighbor['y_pixel'],marker='o',s=34,facecolors='none',edgecolors='lime')
        ax.set_title(f"Sector {item.get('sector')} / Cam {item.get('camera')} / CCD {item.get('ccd')}")
        ax.set_xlabel('Pixel column'); ax.set_ylabel('Pixel row')
    fig.suptitle('TIC 11521787: TESS target-pixel screening\ncyan = aperture; green circles = Gaia DR3; white + = TIC coordinate')
    fig.tight_layout(); fig.savefig(plot_path,dpi=160,bbox_inches='tight'); plt.close(fig)

run_status='completed' if downloaded and all(item.get('status')!='audit_failed' for item in product_results) else 'needs_review'
if gaia.get('status')!='query_succeeded': run_status='completed_with_catalog_warning' if downloaded else 'needs_review'
summary={
    'created_utc':datetime.now(timezone.utc).isoformat(),'source_id':TARGET,'run_status':run_status,
    'features_input':str(FEATURE_PATH),'mast_status':mast_status['status'],'gaia_status':gaia['status'],
    'pixel_products_downloaded':len(downloaded),'pixel_products_audited':sum(item.get('status')!='audit_failed' for item in product_results),
    'gaia_rows_returned':gaia.get('record_count_returned'),
    'outputs':[str(report_path),str(summary_csv),str(neighbors_csv),str(OUTPUT/'day18_target_pixel_summary.json')]+([str(plot_path)] if plot_path.exists() else []),
    'interpretation_limits':report['interpretation_limits'],
}
summary_json=OUTPUT/'day18_target_pixel_summary.json'
summary_json.write_text(json.dumps(clean_json(summary),indent=2,allow_nan=False),encoding='utf-8')
print('\nFinal status:',run_status)
print(json.dumps(summary,indent=2))
if summary_rows: display(pd.DataFrame(summary_rows))
print('\nSaved outputs under:',OUTPUT)